# PoTeC comprehension & background-knowledge questions

For each of the 12 texts (6 biology `b0`-`b5`, 6 physics `p0`-`p5`) this notebook shows:

- the **3 text-comprehension (TQ)** and **3 background-knowledge (BQ)** questions, their 4 options, and the marked correct answer;
- **how the 75 subjects answered** them — percent correct, broken down by reader major (biology vs physics students).

Sources:
- `stimuli/stimuli/stimuli.tsv` — question text, options, correct-answer index.
- `participants/participant_response_accuracy.tsv` — per reader×text×question correctness (0/1).

Note: the eye-tracking cohort's file records only **whether** each question was answered correctly, not which wrong option was chosen (chosen options live in the separate online-survey file). So "how subjects answered" = correctness rate.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# project root on the path so `src` imports work from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import config

MAJORS = {0: "biology students", 1: "physics students"}
QIDS = [f"{t}_{n}" for t in ("tq", "bq") for n in (1, 2, 3)]  # tq_1..tq_3, bq_1..bq_3

stim = pd.read_csv(config.POTEC_DIR / "stimuli/stimuli/stimuli.tsv", sep="\t")
resp = pd.read_csv(config.POTEC_DIR / "participants/participant_response_accuracy.tsv", sep="\t")
print("texts:", sorted(stim.text_id), "  readers:", resp.reader_id.nunique(), "  rows:", len(resp))

## Reshape the per-question answers

Melt the wide `acc_tq_1..3` / `acc_bq_1..3` correctness columns into one row per (reader, text, question), tagged with the reader's major.

In [ ]:
acc_cols = [f"acc_{q}" for q in QIDS]
long = resp.melt(
    id_vars=["reader_id", "text_id", "text_domain", "reader_discipline_numeric"],
    value_vars=acc_cols, var_name="qid", value_name="correct",
)
long["qid"] = long["qid"].str.replace("acc_", "", regex=False)
long["major"] = long["reader_discipline_numeric"].map(MAJORS)
long.head()

In [ ]:
def question_meta(text_id, qid):
    """(question text, [4 options], correct option index 1-4) for one text+question."""
    row = stim[stim.text_id == text_id].iloc[0]
    opts = [row[f"{qid}_option{i}"] for i in range(1, 5)]
    return row[qid], opts, int(row[f"correct_ans_{qid}"])


def answer_breakdown(text_id):
    """Per-question correctness for one text: overall + by reader major (% correct)."""
    sub = long[long.text_id == text_id]
    overall = sub.groupby("qid")["correct"].agg(["mean", "count"])
    by_major = sub.pivot_table(index="qid", columns="major", values="correct", aggfunc="mean")
    out = pd.DataFrame({
        "type": [q.split("_")[0].upper() for q in QIDS],
        "% correct (all)": (overall["mean"] * 100).round(0),
        "% bio students": (by_major.get("biology students") * 100).round(0),
        "% phys students": (by_major.get("physics students") * 100).round(0),
        "n": overall["count"].astype(int),
    }, index=QIDS)
    return out

In [ ]:
def show_text(text_id):
    """Render every question for a text, the marked correct option, and how subjects scored."""
    row = stim[stim.text_id == text_id].iloc[0]
    md = [f"### `{text_id}` — {row.text_domain} — **{row.headline}**\n"]
    for qid in QIDS:
        qtext, opts, correct = question_meta(text_id, qid)
        md.append(f"**{qid.upper()}** — {qtext}")
        for i, opt in enumerate(opts, start=1):
            mark = " ✅" if i == correct else ""
            md.append(f"- ({i}) {opt}{mark}")
        md.append("")
    display(Markdown("\n".join(md)))
    display(answer_breakdown(text_id))


# Example: a biology text. Change to any of b0-b5 / p0-p5.
show_text("b0")

## Browse any text

Change `text_id` below to any of `b0`-`b5` (biology) or `p0`-`p5` (physics).

In [ ]:
text_id = "p0"
show_text(text_id)

## All texts at once

In [ ]:
for tid in sorted(stim.text_id):
    show_text(tid)

## Summary: % correct per text × question, by reader major

One tidy table over all 72 questions (12 texts × 6). The in-domain reader major should lead its own domain's questions.

In [ ]:
summary = (
    long.groupby(["text_id", "text_domain", "qid", "major"])["correct"]
    .mean().mul(100).round(0)
    .unstack("major").reset_index()
)
summary["qtype"] = summary["qid"].str.split("_").str[0].str.upper()
summary.to_csv(ROOT / "results_question_accuracy_by_major.csv", index=False)
print("wrote results_question_accuracy_by_major.csv")
summary.head(12)

## Heatmap: question correctness across texts

Per text, mean % correct on its TQ vs BQ questions, split by reader major — the in-domain advantage at a glance.

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# mean correctness per text, by question-type and reader major
long["qtype"] = long["qid"].str.split("_").str[0].str.upper()
grid = (
    long.groupby(["text_id", "qtype", "major"])["correct"].mean().mul(100)
    .unstack(["qtype", "major"])
    .reindex(sorted(stim.text_id))
)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(grid.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
ax.set_xticks(range(len(grid.columns)))
ax.set_xticklabels([f"{a}\n{b.split()[0]}" for a, b in grid.columns], fontsize=8)
ax.set_yticks(range(len(grid.index)))
ax.set_yticklabels(grid.index)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                color="white" if v < 55 else "black", fontsize=8)
ax.set_title("% correct per text (question type × reader major)")
fig.colorbar(im, ax=ax, label="% correct")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_question_accuracy_heatmap.png", dpi=150, bbox_inches="tight")
print("wrote figures/eda_question_accuracy_heatmap.png")
plt.show()